# IQ-235 — full source-checkpoint TPU inference
Select **Runtime → Change runtime type → TPU**. This runs the actual 235-layer model with BF16 expert streaming. The stored checkpoint uses the Mamba/Llama lexical foundation, not GLM-5.3 token IDs. No paid resources or training are launched.

In [ ]:
!git clone -b feat/iq235-colab-tpu-streaming https://github.com/Harqer/IQ.git
%cd IQ
%pip install -U torch 'torch_xla[tpu]' 'huggingface_hub==1.14.0' 'transformers==5.17.0' safetensors

If package installation replaces the preinstalled PyTorch, restart the session before running inference. The next cell runs in a fresh subprocess. Add `HF_TOKEN` in the notebook Secrets panel; access to the bucket and official source tokenizer is required for text mode.

## Optional: direct mounted checkpoint
If hf-mount is installed in the runtime, run the next cell before inference. It uses a read-only NFS mount by default. On Colab, mount permissions are not guaranteed. If the mount fails, the inference cell uses the bucket API. No full checkpoint download is required in either mode.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MOUNT = Path("/content/iq-weights")
RUN = MOUNT / "IQ/runs/glm53-full78-local-20261025a"
if shutil.which("hf-mount"):
    MOUNT.mkdir(exist_ok=True)
    if not (RUN / "iq_config.json").is_file():
        subprocess.run(["hf-mount", "start", "--read-only", "--cache-size",
                        "10000000000", "bucket", "kofecheckmate/iq-glm53-all256",
                        str(MOUNT)], check=False)
    print("Mount visible:", (RUN / "iq_config.json").is_file())
else:
    print("hf-mount not installed; see docs/IQ235_COLAB_TPU.md for installation.")

In [ ]:
%cd /content/IQ
import os, subprocess, sys
from pathlib import Path
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
mount = Path("/content/iq-weights")
run = mount / "IQ/runs/glm53-full78-local-20261025a"
cmd = [sys.executable, "-u", "scripts/iq235_colab_tpu.py",
       "--prompt", "The capital of France is", "--max-new-tokens", "2"]
if (run / "iq_config.json").is_file():
    cmd += ["--mount-dir", str(mount)]
print("Transport:", "hf-mount" if "--mount-dir" in cmd else "bucket API")
subprocess.run(cmd, check=True)

If the official tokenizer is gated but the bucket is accessible, run the cell below instead for a real token-ID diagnostic. This runs the same full 235-layer model but does not produce decoded text.

In [ ]:
# Uncomment to use source-token IDs instead of the gated tokenizer.
# !python scripts/iq235_colab_tpu.py --token-ids 128000 32 --max-new-tokens 2